# SQL practice 7: types, casting, numbers and strings

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/sql/sql-practice-07-types-casting-and-strings.ipynb)

14 exercises on casting, number parsing, division and rounding, string cleanup, regular expressions and hashing. Each exercise names the [reference entries](sql-reference-07-types-casting-and-strings.ipynb) it practises, shows the exact expected output, and hides a reference solution you can reveal when you are done or stuck.

*Last reviewed: November 2026, against Apache Spark 4.0 (Spark SQL).*

In [ ]:
# Setup: installs the packages this notebook needs when they are missing (e.g. in Colab).
!pip install -q pyspark~=4.0.0

## Setup and sample data

A local Spark session, the shared tables as temporary views: `customers`, `orders`, `order_items`, `products`, `events` and `employees` (customer 4 has no region, customers 7 and 8 never ordered, `SKU-4014` has no product row, one event is an exact retry, two pairs of employees tie on salary, Dana out-earns her manager and Jo has no department), and the helpers from [SQL reference 7](sql-reference-07-types-casting-and-strings.ipynb#setup-and-sample-data). Exercises that need other data create it in their input cell.

In [ ]:
import os
import sys
import tempfile
import time

from pyspark.sql import SparkSession

os.environ["PYSPARK_PYTHON"] = sys.executable  # Python workers use this notebook's interpreter
os.environ["TZ"] = "UTC"  # the sample timestamps are UTC
time.tzset()
WORK_DIR = tempfile.mkdtemp(prefix="pyspark-")  # scratch space for the files the lessons write

spark = (
    SparkSession.builder.master("local[2]")
    .config("spark.sql.shuffle.partitions", "4")  # the default of 200 is far too many for tiny data
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.warehouse.dir", os.path.join(WORK_DIR, "warehouse"))
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

In [ ]:
# Shared sample tables as temporary views. Each statement is ordinary Spark SQL.
TABLES = {
    "customers": """CREATE OR REPLACE TEMP VIEW customers AS SELECT * FROM VALUES
  (1, 'Acme Corp',      'North', DATE'2024-01-03'),
  (2, 'Bright Retail',  'North', DATE'2024-02-11'),
  (3, 'Cyan Traders',   'South', DATE'2024-03-15'),
  (4, 'Delta Grocers',  NULL,    DATE'2024-05-02'),   -- region never arrived from the CRM
  (5, 'Echo Supplies',  'West',  DATE'2024-06-20'),
  (6, 'Foxtrot Foods',  'West',  DATE'2024-08-01'),
  (7, 'Gamma Sports',   'East',  DATE'2024-09-14'),   -- 7 and 8 have never ordered
  (8, 'Hotel Homeware', 'East',  DATE'2024-11-05')
AS customers(customer_id, name, region, signup_date)""",
    "orders": """CREATE OR REPLACE TEMP VIEW orders AS SELECT order_id, customer_id, status, order_ts, CAST(amount AS DECIMAL(12,2)) AS amount FROM VALUES
  (1,  1, 'paid',     TIMESTAMP'2026-01-05 09:15:00', 120.00),
  (2,  1, 'shipped',  TIMESTAMP'2026-01-05 10:05:00', 84.50),
  (3,  1, 'returned', TIMESTAMP'2026-01-05 11:40:00', 30.00),
  (4,  2, 'paid',     TIMESTAMP'2026-01-06 15:20:00', 260.00),
  (5,  2, 'placed',   TIMESTAMP'2026-01-07 08:05:00', 55.25),
  (6,  3, 'shipped',  TIMESTAMP'2026-01-07 17:45:00', 149.00),
  (7,  3, 'paid',     TIMESTAMP'2026-01-09 12:10:00', 310.00),
  (8,  4, 'returned', TIMESTAMP'2026-01-10 19:30:00', 89.90),
  (9,  4, 'shipped',  TIMESTAMP'2026-01-12 08:55:00', 1180.00),
  (10, 5, 'paid',     TIMESTAMP'2026-01-12 13:00:00', 74.00),
  (11, 6, 'placed',   TIMESTAMP'2026-01-13 10:25:00', 210.00),
  (12, 6, 'shipped',  TIMESTAMP'2026-01-14 00:30:00', 96.40)   -- just after midnight UTC
AS orders(order_id, customer_id, status, order_ts, amount)""",
    "order_items": """CREATE OR REPLACE TEMP VIEW order_items AS SELECT order_id, sku, qty, CAST(unit_price AS DECIMAL(8,2)) AS unit_price FROM VALUES
  (1, 'SKU-4001', 2, 12.50), (1, 'SKU-4007', 1, 95.00), (2, 'SKU-4002', 1, 34.50), (2, 'SKU-4009', 2, 25.00),
  (3, 'SKU-4002', 1, 30.00), (4, 'SKU-4003', 1, 149.00), (4, 'SKU-4004', 2, 55.50), (5, 'SKU-4005', 1, 55.25),
  (6, 'SKU-4003', 1, 149.00), (7, 'SKU-4006', 1, 189.00), (7, 'SKU-4008', 1, 121.00), (8, 'SKU-4010', 1, 89.90),
  (9, 'SKU-4011', 20, 59.00), (10, 'SKU-4012', 2, 37.00), (11, 'SKU-4013', 1, 210.00),
  (12, 'SKU-4014', 1, 41.40), (12, 'SKU-4005', 1, 55.00)
AS order_items(order_id, sku, qty, unit_price)""",
    "products": """CREATE OR REPLACE TEMP VIEW products AS SELECT sku, category, CAST(list_price AS DECIMAL(8,2)) AS list_price FROM VALUES
  ('SKU-4001', 'Kitchen', 12.50), ('SKU-4002', 'Kitchen', 34.50), ('SKU-4003', 'Garden', 149.00),
  ('SKU-4004', 'Garden', 55.50), ('SKU-4005', 'Pantry', 55.25), ('SKU-4006', 'Outdoor', 189.00),
  ('SKU-4007', 'Kitchen', 95.00), ('SKU-4008', 'Outdoor', 121.00), ('SKU-4009', 'Pantry', 25.00),
  ('SKU-4010', 'Garden', 89.90), ('SKU-4011', 'Pantry', 59.00), ('SKU-4012', 'Kitchen', 37.00),
  ('SKU-4013', 'Outdoor', 210.00)   -- SKU-4014 has no product row
AS products(sku, category, list_price)""",
    "events": """CREATE OR REPLACE TEMP VIEW events AS SELECT * FROM VALUES
  (1, 'view', TIMESTAMP'2026-01-06 09:00:00'), (1, 'add_to_cart', TIMESTAMP'2026-01-06 09:02:00'),
  (1, 'purchase', TIMESTAMP'2026-01-06 09:10:00'), (2, 'view', TIMESTAMP'2026-01-07 10:00:00'),
  (2, 'purchase', TIMESTAMP'2026-01-07 10:40:00'), (3, 'view', TIMESTAMP'2026-01-06 11:15:00'),
  (3, 'view', TIMESTAMP'2026-01-08 11:20:00'), (4, 'add_to_cart', TIMESTAMP'2026-01-09 20:05:00'),
  (4, 'purchase', TIMESTAMP'2026-01-10 08:30:00'), (5, 'view', TIMESTAMP'2026-01-11 14:45:00'),
  (5, 'view', TIMESTAMP'2026-01-12 09:00:00'), (5, 'purchase', TIMESTAMP'2026-01-12 12:30:00'),
  (5, 'view', TIMESTAMP'2026-01-12 09:00:00')   -- an SDK retry committed this event twice
AS events(user_id, event, ts)""",
    "employees": """CREATE OR REPLACE TEMP VIEW employees AS SELECT * FROM VALUES
  (1,  'Ada',    'Engineering', CAST(NULL AS INT), 185000, DATE'2018-03-01'),
  (2,  'Brett',  'Engineering', 1,    150000, DATE'2019-07-15'),
  (3,  'Chen',   'Engineering', 1,    150000, DATE'2020-01-06'),
  (4,  'Dana',   'Engineering', 2,    155000, DATE'2022-09-12'),   -- out-earns her manager
  (5,  'Eli',    'Sales',       1,    130000, DATE'2019-02-01'),
  (6,  'Farah',  'Sales',       5,    95000,  DATE'2021-05-17'),
  (7,  'Gus',    'Sales',       5,    95000,  DATE'2023-01-09'),
  (8,  'Hana',   'Finance',     1,    140000, DATE'2017-11-20'),
  (9,  'Ivan',   'Finance',     8,    88000,  DATE'2024-04-01'),
  (10, 'Jo',     NULL,          8,    70000,  DATE'2025-06-30')   -- department not assigned yet
AS employees(emp_id, name, dept, manager_id, salary, hire_date)""",
}
for statement in TABLES.values():
    spark.sql(statement)

import contextlib
import io
import logging
import re


def show(query, n=50, **args):
    """Run a query and print up to n rows without truncating values. Keyword arguments bind :name parameters."""
    spark.sql(query, args=args or None).show(n, truncate=False)


def plan(query, mode="formatted"):
    """The plan EXPLAIN prints for a query, as text."""
    return spark.sql(f"EXPLAIN {mode.upper()} {query}").first()[0]


@contextlib.contextmanager
def expected_error():
    """Run a block that is meant to fail and print the error class instead of a stack trace."""
    spark.sparkContext.setLogLevel("OFF")
    for logger in ("DataFrameQueryContextLogger", "SQLQueryContextLogger"):
        logging.getLogger(logger).setLevel(logging.CRITICAL)
    try:
        yield
    except Exception as error:
        lines = [line for line in str(error).splitlines() if line.strip()]
        message = next((line for line in lines if re.search(r"\[[A-Z][A-Z0-9_.]+\]", line)), lines[0] if lines else "")
        print(type(error).__name__ + ":", message.strip().split(". ")[0][:120])
    finally:
        spark.sparkContext.setLogLevel("ERROR")

## Exercises

9 easy, 4 medium, 1 hard. Write your query in the empty cell under each exercise, run it with `show(q)` or `spark.sql(q).show()`, and compare it with the expected output (columns, values and row order).

| # | exercise | difficulty | practises |
| --- | --- | --- | --- |
| 1 | [Type a raw feed](#exercise-1-type-a-raw-feed) | Easy | [`CAST`, `::` and `TRY_CAST`](sql-reference-07-types-casting-and-strings.ipynb#cast--and-try_cast) |
| 2 | [Formatted numbers to decimals](#exercise-2-formatted-numbers-to-decimals) | Medium | [`CAST`, `::` and `TRY_CAST`](sql-reference-07-types-casting-and-strings.ipynb#cast--and-try_cast) |
| 3 | [Exact unit prices](#exercise-3-exact-unit-prices) | Easy | [Arithmetic and division](sql-reference-07-types-casting-and-strings.ipynb#arithmetic-and-division) |
| 4 | [Round half up versus half even](#exercise-4-round-half-up-versus-half-even) | Easy | [Rounding](sql-reference-07-types-casting-and-strings.ipynb#rounding) |
| 5 | [Email domains](#exercise-5-email-domains) | Easy | [Core string functions](sql-reference-07-types-casting-and-strings.ipynb#core-string-functions) |
| 6 | [Normalise phone numbers](#exercise-6-normalise-phone-numbers) | Medium | [Regular expressions](sql-reference-07-types-casting-and-strings.ipynb#regular-expressions) |
| 7 | [Clean up messy names](#exercise-7-clean-up-messy-names) | Easy | [Core string functions](sql-reference-07-types-casting-and-strings.ipynb#core-string-functions), [Regular expressions](sql-reference-07-types-casting-and-strings.ipynb#regular-expressions) |
| 8 | [Customer codes](#exercise-8-customer-codes) | Easy | [Core string functions](sql-reference-07-types-casting-and-strings.ipynb#core-string-functions) |
| 9 | [Order numbers mentioned in notes](#exercise-9-order-numbers-mentioned-in-notes) | Medium | [Regular expressions](sql-reference-07-types-casting-and-strings.ipynb#regular-expressions) |
| 10 | [Mask card numbers](#exercise-10-mask-card-numbers) | Easy | [Regular expressions](sql-reference-07-types-casting-and-strings.ipynb#regular-expressions) |
| 11 | [Pseudonymous keys](#exercise-11-pseudonymous-keys) | Easy | [Formatting, hashing and encoding](sql-reference-07-types-casting-and-strings.ipynb#formatting-hashing-and-encoding) |
| 12 | [Distinct names ignoring case](#exercise-12-distinct-names-ignoring-case) | Easy | [Collations and case-insensitive comparison](sql-reference-07-types-casting-and-strings.ipynb#collations-and-case-insensitive-comparison) |
| 13 | [Campaign parameters from URLs](#exercise-13-campaign-parameters-from-urls) | Medium | [Formatting, hashing and encoding](sql-reference-07-types-casting-and-strings.ipynb#formatting-hashing-and-encoding) |
| 14 | [Validate email addresses](#exercise-14-validate-email-addresses) | Hard | [Regular expressions](sql-reference-07-types-casting-and-strings.ipynb#regular-expressions), [Core string functions](sql-reference-07-types-casting-and-strings.ipynb#core-string-functions) |

## Exercise 1: Type a raw feed

**Difficulty:** Easy · about 5 min · **Practises:** [`CAST`, `::` and `TRY_CAST`](sql-reference-07-types-casting-and-strings.ipynb#cast--and-try_cast)

*Landing tables are all strings; staging casts them and keeps bad values visible.*

Using `raw_orders` below, return `order_id` (INT), `amount` (DECIMAL(10,2), NULL when invalid), `paid` (BOOLEAN) and `amount_ok` (whether the amount parsed), ordered by `order_id`.

In [ ]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW raw_orders AS SELECT * FROM VALUES
  ('3', '19.99', 'true'), ('1', '5', 'false'), ('2', 'abc', 'true'), ('4', '12.345', 'FALSE')
AS raw_orders(order_id, amount, paid)
""")
show("SELECT *, typeof(order_id) AS order_id_type FROM raw_orders")

**Expected output**

| order_id | amount | paid | amount_ok |
| --- | --- | --- | --- |
| 1 | 5.00 | false | true |
| 2 | null | true | false |
| 3 | 19.99 | true | true |
| 4 | 12.35 | false | true |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT CAST(order_id AS INT) AS order_id,
           try_cast(amount AS DECIMAL(10,2)) AS amount,
           CAST(paid AS BOOLEAN) AS paid,
           try_cast(amount AS DECIMAL(10,2)) IS NOT NULL AS amount_ok
    FROM raw_orders
    ORDER BY order_id
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 2: Formatted numbers to decimals

**Difficulty:** Medium · about 5 min · **Practises:** [`CAST`, `::` and `TRY_CAST`](sql-reference-07-types-casting-and-strings.ipynb#cast--and-try_cast)

*Exports from spreadsheets and finance systems include currency symbols and separators.*

Using `invoices` below, return `invoice` and `total` as a decimal parsed from `total_text` (format like `$1,234.50`), NULL when it does not match, ordered by `invoice`.

In [ ]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW invoices AS SELECT * FROM VALUES
  ('INV-1', '$1,234.50'), ('INV-2', '$99.00'), ('INV-3', '1,000'), ('INV-4', '$2,000.75')
AS invoices(invoice, total_text)
""")

**Expected output**

| invoice | total |
| --- | --- |
| INV-1 | 1234.50 |
| INV-2 | 99.00 |
| INV-3 | null |
| INV-4 | 2000.75 |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT invoice, try_to_number(total_text, '$9,999.99') AS total
    FROM invoices
    ORDER BY invoice
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 3: Exact unit prices

**Difficulty:** Easy · about 5 min · **Practises:** [Arithmetic and division](sql-reference-07-types-casting-and-strings.ipynb#arithmetic-and-division)

*Division semantics differ between engines; be explicit.*

Return `order_id`, `sku`, `qty`, `line_total` and `units_per_100` (how many full units 100 currency units would buy at this unit price, as an integer) for order 1 and 2 lines, ordered by `order_id`, `sku`.

**Expected output**

| order_id | sku | qty | line_total | units_per_100 |
| --- | --- | --- | --- | --- |
| 1 | SKU-4001 | 2 | 25.00 | 8 |
| 1 | SKU-4007 | 1 | 95.00 | 1 |
| 2 | SKU-4002 | 1 | 34.50 | 2 |
| 2 | SKU-4009 | 2 | 50.00 | 4 |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT order_id, sku, qty, qty * unit_price AS line_total,
           CAST(100 div unit_price AS INT) AS units_per_100
    FROM order_items
    WHERE order_id IN (1, 2)
    ORDER BY order_id, sku
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 4: Round half up versus half even

**Difficulty:** Easy · about 3 min · **Practises:** [Rounding](sql-reference-07-types-casting-and-strings.ipynb#rounding)

*Financial reports must state which rounding rule they use.*

Using `prices` below, return `price`, `half_up` and `half_even` rounded to 1 decimal, and `to_tens` rounded to the nearest 10, ordered by `price`.

In [ ]:
spark.sql("CREATE OR REPLACE TEMP VIEW prices AS SELECT CAST(p AS DECIMAL(6,2)) AS price FROM VALUES (2.25), (2.35), (14.95), (-1.05), (125.50) AS t(p)")

**Expected output**

| price | half_up | half_even | to_tens |
| --- | --- | --- | --- |
| -1.05 | -1.1 | -1.0 | 0 |
| 2.25 | 2.3 | 2.2 | 0 |
| 2.35 | 2.4 | 2.4 | 0 |
| 14.95 | 15.0 | 15.0 | 10 |
| 125.50 | 125.5 | 125.5 | 130 |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT price, round(price, 1) AS half_up, bround(price, 1) AS half_even, round(price, -1) AS to_tens
    FROM prices
    ORDER BY price
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 5: Email domains

**Difficulty:** Easy · about 5 min · **Practises:** [Core string functions](sql-reference-07-types-casting-and-strings.ipynb#core-string-functions)

*Grouping signups by email provider.*

Using `signups` below, return `domain` (lower-cased part after `@`, trimmed) and `signups`, ordered by `signups` descending, then `domain`.

In [ ]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW signups AS SELECT * FROM VALUES
  ('ana@Gmail.com'), ('ben@example.org'), ('cy@gmail.com '), ('dee@Example.org'), ('eve@proton.me')
AS signups(email)
""")

**Expected output**

| domain | signups |
| --- | --- |
| example.org | 2 |
| gmail.com | 2 |
| proton.me | 1 |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT lower(trim(split_part(email, '@', 2))) AS domain, count(*) AS signups
    FROM signups
    GROUP BY lower(trim(split_part(email, '@', 2)))
    ORDER BY signups DESC, domain
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 6: Normalise phone numbers

**Difficulty:** Medium · about 10 min · **Practises:** [Regular expressions](sql-reference-07-types-casting-and-strings.ipynb#regular-expressions)

*Matching customers across systems needs one phone format.*

Using `phones` below, return `customer_id` and `phone`: the last 10 digits of the number with every non-digit removed, formatted `XXX-XXX-XXXX`; NULL when fewer than 10 digits remain. Order by `customer_id`.

In [ ]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW phones AS SELECT * FROM VALUES
  (1, '(555) 010-2030'), (2, '+1 555.010.4050'), (3, '010-20'), (4, '5550106070 ext')
AS phones(customer_id, raw)
""")

**Expected output**

| customer_id | phone |
| --- | --- |
| 1 | 555-010-2030 |
| 2 | 555-010-4050 |
| 3 | null |
| 4 | 555-010-6070 |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = r"""
    WITH digits AS (
        SELECT customer_id, regexp_replace(raw, '[^0-9]', '') AS d FROM phones
    )
    SELECT customer_id,
           CASE WHEN length(d) >= 10
                THEN regexp_replace(right(d, 10), '^(\\d{3})(\\d{3})(\\d{4})$', '$1-$2-$3') END AS phone
    FROM digits
    ORDER BY customer_id
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 7: Clean up messy names

**Difficulty:** Easy · about 5 min · **Practises:** [Core string functions](sql-reference-07-types-casting-and-strings.ipynb#core-string-functions), [Regular expressions](sql-reference-07-types-casting-and-strings.ipynb#regular-expressions)

*Free-text names arrive with stray spaces and random capitalisation.*

Using `raw_names` below, return `id` and `name` trimmed, with runs of spaces collapsed to one and each word capitalised, ordered by `id`.

In [ ]:
spark.sql("CREATE OR REPLACE TEMP VIEW raw_names AS SELECT * FROM VALUES (1, '  aDA   lovelace '), (2, 'GRACE HOPPER'), (3, 'alan  m.  turing') AS t(id, name)")

**Expected output**

| id | name |
| --- | --- |
| 1 | Ada Lovelace |
| 2 | Grace Hopper |
| 3 | Alan M. Turing |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT id, initcap(regexp_replace(trim(name), ' +', ' ')) AS name
    FROM raw_names
    ORDER BY id
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 8: Customer codes

**Difficulty:** Easy · about 3 min · **Practises:** [Core string functions](sql-reference-07-types-casting-and-strings.ipynb#core-string-functions)

*Fixed-width identifiers for downstream systems.*

Return `customer_id` and `code`: `C-` followed by the id left-padded with zeros to 5 digits, then `-` and the first three letters of the region in upper case (`UNK` when missing). Order by `customer_id`.

**Expected output**

| customer_id | code |
| --- | --- |
| 1 | C-00001-NOR |
| 2 | C-00002-NOR |
| 3 | C-00003-SOU |
| 4 | C-00004-UNK |
| 5 | C-00005-WES |
| 6 | C-00006-WES |
| 7 | C-00007-EAS |
| 8 | C-00008-EAS |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT customer_id,
           concat('C-', lpad(CAST(customer_id AS STRING), 5, '0'), '-', upper(left(coalesce(region, 'unknown'), 3))) AS code
    FROM customers
    ORDER BY customer_id
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 9: Order numbers mentioned in notes

**Difficulty:** Medium · about 5 min · **Practises:** [Regular expressions](sql-reference-07-types-casting-and-strings.ipynb#regular-expressions)

*Support tickets reference orders in free text.*

Using `tickets` below, return `ticket_id`, `order_ids` (all numbers that follow `#`, as an array of integers) and `mentions`, ordered by `ticket_id`.

In [ ]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW tickets AS SELECT * FROM VALUES
  (1, 'Refund #12 please, also check #9'), (2, 'Where is my parcel?'), (3, 'Duplicate charge on #1180')
AS tickets(ticket_id, body)
""")

**Expected output**

| ticket_id | order_ids | mentions |
| --- | --- | --- |
| 1 | [12, 9] | 2 |
| 2 | [] | 0 |
| 3 | [1180] | 1 |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = r"""
    SELECT ticket_id,
           transform(regexp_extract_all(body, '#(\\d+)', 1), x -> CAST(x AS INT)) AS order_ids,
           regexp_count(body, '#\\d+') AS mentions
    FROM tickets
    ORDER BY ticket_id
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 10: Mask card numbers

**Difficulty:** Easy · about 5 min · **Practises:** [Regular expressions](sql-reference-07-types-casting-and-strings.ipynb#regular-expressions)

*Never show full card numbers in reports.*

Using `cards` below, return `card_id` and `masked`: every digit except the last four replaced with `*`, keeping separators, ordered by `card_id`.

In [ ]:
spark.sql("CREATE OR REPLACE TEMP VIEW cards AS SELECT * FROM VALUES (1, '4111-1111-1111-1234'), (2, '5500 0000 0000 0004'), (3, '378282246310005') AS t(card_id, number)")

**Expected output**

| card_id | masked |
| --- | --- |
| 1 | ****-****-****-1234 |
| 2 | **** **** **** 0004 |
| 3 | ***********0005 |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = r"""
    SELECT card_id, regexp_replace(number, '\\d(?=(?:\\D*\\d){4})', '*') AS masked
    FROM cards
    ORDER BY card_id
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 11: Pseudonymous keys

**Difficulty:** Easy · about 3 min · **Practises:** [Formatting, hashing and encoding](sql-reference-07-types-casting-and-strings.ipynb#formatting-hashing-and-encoding)

*Analytics joins on a stable hash instead of the raw email.*

Using `signups` from exercise 5 (re-created below), return `email_key` (the first 16 hex characters of SHA-256 of the trimmed, lower-cased email) and `signups` per key, ordered by `email_key`.

In [ ]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW signups AS SELECT * FROM VALUES
  ('ana@Gmail.com'), ('ANA@gmail.com '), ('ben@example.org')
AS signups(email)
""")

**Expected output**

| email_key | signups |
| --- | --- |
| 26e2d102014008d9 | 2 |
| 9de4dcc67240d506 | 1 |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT left(sha2(lower(trim(email)), 256), 16) AS email_key, count(*) AS signups
    FROM signups
    GROUP BY left(sha2(lower(trim(email)), 256), 16)
    ORDER BY email_key
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 12: Distinct names ignoring case

**Difficulty:** Easy · about 3 min · **Practises:** [Collations and case-insensitive comparison](sql-reference-07-types-casting-and-strings.ipynb#collations-and-case-insensitive-comparison)

*Duplicate detection across sources that disagree on capitalisation.*

Using `vendors` below, return `vendor` (the alphabetically first spelling of each group) and `spellings`, grouping names case-insensitively after trimming, ordered by `vendor`.

In [ ]:
spark.sql("CREATE OR REPLACE TEMP VIEW vendors AS SELECT * FROM VALUES ('Acme Corp'), ('ACME CORP'), (' acme corp'), ('Beta LLC'), ('beta llc') AS t(name)")

**Expected output**

| vendor | spellings |
| --- | --- |
| ACME CORP | 3 |
| Beta LLC | 2 |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT min(trim(name)) AS vendor, count(*) AS spellings
    FROM vendors
    GROUP BY lower(trim(name))
    ORDER BY vendor
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 13: Campaign parameters from URLs

**Difficulty:** Medium · about 5 min · **Practises:** [Formatting, hashing and encoding](sql-reference-07-types-casting-and-strings.ipynb#formatting-hashing-and-encoding)

*Marketing attribution starts with UTM parameters.*

Using `visits` below, return `visit_id`, `source` (the `utm_source` query parameter, `direct` when absent) and `path` (the URL path), ordered by `visit_id`.

In [ ]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW visits AS SELECT * FROM VALUES
  (1, 'https://shop.example/garden?utm_source=newsletter&utm_medium=email'),
  (2, 'https://shop.example/kitchen/knives'),
  (3, 'https://shop.example/?ref=x&utm_source=google')
AS visits(visit_id, url)
""")

**Expected output**

| visit_id | source | path |
| --- | --- | --- |
| 1 | newsletter | /garden |
| 2 | direct | /kitchen/knives |
| 3 | google | / |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT visit_id,
           coalesce(parse_url(url, 'QUERY', 'utm_source'), 'direct') AS source,
           parse_url(url, 'PATH') AS path
    FROM visits
    ORDER BY visit_id
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 14: Validate email addresses

**Difficulty:** Hard · about 15 min · **Practises:** [Regular expressions](sql-reference-07-types-casting-and-strings.ipynb#regular-expressions), [Core string functions](sql-reference-07-types-casting-and-strings.ipynb#core-string-functions)

*Data quality rules classify bad values instead of dropping them.*

Using `contacts` below, return `id`, `email` and `issue`: `missing` (NULL or blank), `has spaces` (inner spaces after trimming), `no @`, `bad domain` (domain part without a dot or ending in a dot), or `ok` for addresses matching `^[^@]+@[^@]+\.[^@.]+$`. Check in that order. Order by `id`.

In [ ]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW contacts AS SELECT * FROM VALUES
  (1, 'ana@example.com'), (2, NULL), (3, ''), (4, 'ben at example.com'), (5, 'cy@localhost'),
  (6, 'dee@example.'), (7, 'eve@mail.example.org'), (8, 'frank.example.com')
AS contacts(id, email)
""")

**Expected output**

| id | email | issue |
| --- | --- | --- |
| 1 | ana@example.com | ok |
| 2 | null | missing |
| 3 |  | missing |
| 4 | ben at example.com | has spaces |
| 5 | cy@localhost | bad domain |
| 6 | dee@example. | bad domain |
| 7 | eve@mail.example.org | ok |
| 8 | frank.example.com | no @ |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = r"""
    SELECT id, email,
           CASE WHEN email IS NULL OR trim(email) = '' THEN 'missing'
                WHEN trim(email) RLIKE ' ' THEN 'has spaces'
                WHEN instr(email, '@') = 0 THEN 'no @'
                WHEN NOT trim(email) RLIKE '^[^@]+@[^@]+\\.[^@.]+$' THEN 'bad domain'
                ELSE 'ok' END AS issue
    FROM contacts
    ORDER BY id
"""
result = spark.sql(q)
result.show(truncate=False)

**Next:** look anything up in [SQL reference 7: types, casting, numbers and strings](sql-reference-07-types-casting-and-strings.ipynb), or try mixed interview problems in [SQL interview drills](sql-13-interview-drills.ipynb).